# ATML PA2: Task 2 - Proximal Policy Optimization (PPO)

This notebook runs all Task 2 experiments on a Kaggle GPU environment (T4 x 1 or P100):
1. **Environment & Assets Setup**
2. **Step 1: Standard PPO Continuation (20 updates from supplied midpoint)**
3. **Step 1: Evaluation on Held-Out Prompts (Reward, KL, Entropy, Length, Peak VRAM, Wall-clock)**
4. **Step 2: Clipping Study ($\epsilon \in \{0.05, 0.20, 0.50\}$ on cached rollout batch + 8-update forks)**
5. **Step 3: KL-Pressure / Reward-Overoptimization Ablation ($\beta_{\text{KL}} \in \{0.0, 0.10, 0.20\}$ 8-update forks)**
6. **Results Packaging & Summary Inspection**

## 1. Verify GPU and Environment

In [ ]:
!nvidia-smi

## 2. Clone Your GitHub Repository
Update `REPO_URL` with your repository URL.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/ATML-PA2-LLM-PostTraining.git"

import os
repo_name = REPO_URL.split('/')[-1].replace('.git', '')
if not os.path.exists(repo_name):
    !git clone $REPO_URL
else:
    %cd $repo_name
    !git pull
    %cd ..

%cd $repo_name
!git status

## 3. Install Dependencies & Validate Course Assets

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Run Task 2 - Step 1: Standard PPO Continuation (20 Updates)
Continues online RLHF from the supplied checkpoints (`checkpoints/ppo_midpoint_policy` and `checkpoints/ppo_midpoint_value`). Logs trajectories of learned reward, reference KL, policy/value losses, entropy, clip fraction, gradient norm, length, peak VRAM, and wall-clock time.

In [ ]:
!python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard

## 5. Run Task 2 - Step 1: Standard PPO Evaluation
Evaluates the continued PPO policy on held-out prompts (`data/rl_prompt_pool_eval.jsonl`, 32 prompts matching instructor acceptance budget at max length 768).

In [ ]:
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard --max-prompts 32

## 6. Run Task 2 - Step 2: Clipping Study ($\epsilon \in \{0.05, 0.20, 0.50\}$)
Measures geometric clipping on the supplied cached rollout batch (`cached/ppo_rollout.pt`), runs matched 8-update continuation forks from the midpoint checkpoint, and logs stability statistics (policy loss & gradient norm variances).

In [ ]:
!python -m task2_ppo.analyze_clipping --config configs/ppo.yaml --max-eval-prompts 32

## 7. Run Task 2 - Step 3: KL-Pressure / Reward-Overoptimization Ablation ($\beta_{\text{KL}} \in \{0.0, 0.10, 0.20\}$)
Runs matched 8-update continuation forks for each $\beta_{\text{KL}}$, evaluates held-out performance, and extracts qualitative cases comparing reward optimization vs. policy drift.

In [ ]:
!python -m task2_ppo.ablate_kl --config configs/ppo.yaml --max-eval-prompts 32

## 8. View and Inspect All Generated Results

In [ ]:
import pandas as pd
import json
from pathlib import Path

res_dir = Path("results/task2_ppo")

print("=== STANDARD PPO EVALUATION ===")
std_json = res_dir / "standard_eval.json"
if std_json.exists():
    print(json.dumps(json.loads(std_json.read_text()), indent=2))

print("\n=== CLIPPING STUDY SUMMARY ===")
clip_csv = res_dir / "clipping_analysis.csv"
if clip_csv.exists():
    display(pd.read_csv(clip_csv))

print("\n=== KL-PRESSURE ABLATION SUMMARY ===")
kl_csv = res_dir / "kl_ablation_summary.csv"
if kl_csv.exists():
    display(pd.read_csv(kl_csv))

## 9. Package Results for Download
Zips all generated JSON/CSV results from `results/task2_ppo` so you can download them directly from Kaggle.

In [ ]:
# Zip results directory (small JSON and CSV logs)
!zip -r task2_results.zip results/task2_ppo
print("task2_results.zip ready in Kaggle working directory!")

# Optional: push results back to your GitHub repository
# !git config --global user.email "your_email@example.com"
# !git config --global user.name "Your Name"
# !git add results/task2_ppo
# !git commit -m "Add Task 2 PPO results and logs"
# !git push